In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [ ]:
# Q11: Annualized revenue loss due to churn
churned_df = df[df['Churn'] == 'Yes']

monthly_loss = churned_df['MonthlyCharges'].sum()
annual_loss  = round(monthly_loss * 12, 2)

print(f'Churned customers:       {len(churned_df)}')
print(f'Monthly revenue loss:    ${monthly_loss:,.2f}')
print(f'Annualized revenue loss: ${annual_loss:,.2f}')

In [ ]:
# Q12: Pareto analysis — Do 20% of telecom customers generate 80% of total revenue
df_pareto = df[df['TotalCharges'] > 0].copy()
df_pareto = df_pareto.sort_values('TotalCharges', ascending=False)

df_pareto['cumulative_revenue'] = df_pareto['TotalCharges'].cumsum()
df_pareto['cumulative_pct']     = df_pareto['cumulative_revenue'] / df_pareto['TotalCharges'].sum() * 100

top_20_pct_cutoff = int(len(df_pareto) * 0.20)
top_20_customers  = df_pareto.head(top_20_pct_cutoff)

print(f'Total customers:              {len(df_pareto)}')
print(f'Top 20% customer count:       {top_20_pct_cutoff}')
print(f'Revenue from top 20%:         ${top_20_customers["TotalCharges"].sum():,.2f}')
print(f'Total revenue:                ${df_pareto["TotalCharges"].sum():,.2f}')
print(f'Top 20% revenue contribution: {round(top_20_customers["TotalCharges"].sum() / df_pareto["TotalCharges"].sum() * 100, 2)}%')

In [4]:
# Q13: ARPU by Contract Type, Internet Service, Payment Method
arpu_contract = df.groupby('Contract')['MonthlyCharges'].mean().round(2).reset_index()
arpu_contract.columns = ['Contract', 'ARPU']
print("ARPU by Contract Type:")
print(arpu_contract.to_string(index=False))

print()

arpu_internet = df.groupby('InternetService')['MonthlyCharges'].mean().round(2).reset_index()
arpu_internet.columns = ['InternetService', 'ARPU']
print("ARPU by Internet Service:")
print(arpu_internet.to_string(index=False))

print()

arpu_payment = df.groupby('PaymentMethod')['MonthlyCharges'].mean().round(2).reset_index()
arpu_payment.columns = ['PaymentMethod', 'ARPU']
print("ARPU by Payment Method:")
print(arpu_payment.to_string(index=False))


ARPU by Contract Type:
      Contract  ARPU
Month-to-month 66.40
      One year 65.05
      Two year 60.77

ARPU by Internet Service:
InternetService  ARPU
            DSL 58.10
    Fiber optic 91.50
             No 21.08

ARPU by Payment Method:
            PaymentMethod  ARPU
Bank transfer (automatic) 67.19
  Credit card (automatic) 66.51
         Electronic check 76.26
             Mailed check 43.92


In [ ]:
# Q14: Revenue leakage report — high charges, low tenure, churned
high_charge_threshold = df['MonthlyCharges'].mean()
low_tenure_threshold  = df['tenure'].quantile(0.25)

leakage = df[
    (df['MonthlyCharges'] > high_charge_threshold) &
    (df['tenure'] <= low_tenure_threshold) &
    (df['Churn'] == 'Yes')
][['customerID', 'Contract', 'InternetService', 'MonthlyCharges', 'tenure', 'TotalCharges']]

print(f'High charge threshold:  ${round(high_charge_threshold, 2)}')
print(f'Low tenure threshold:   {int(low_tenure_threshold)} months')
print(f'Revenue leakage customers: {len(leakage)}')
print(f'Monthly revenue lost:   ${leakage["MonthlyCharges"].sum():,.2f}')
leakage.head(10)

In [ ]:
# Q15: Customers paying more than 1.5 standard deviations above mean monthly charge
mean_charge = df['MonthlyCharges'].mean()
std_charge  = df['MonthlyCharges'].std()
threshold   = mean_charge + 1.5 * std_charge

high_payers = df[df['MonthlyCharges'] > threshold][['customerID', 'MonthlyCharges', 'Contract', 'InternetService', 'Churn']]

print(f'Mean monthly charge:     ${round(mean_charge, 2)}')
print(f'Standard deviation:      ${round(std_charge, 2)}')
print(f'Threshold:               ${round(threshold, 2)}')
print(f'High paying customers:   {len(high_payers)}')
print(f'Churn rate among them:   {round((high_payers["Churn"] == "Yes").sum() / len(high_payers) * 100, 2)}%')
high_payers.head(10)